# Variational autoencoder - advanced walkthrough

Run every cell from top to bottom. The notebook prints intermediate values and draws visualizations so the math stays visible.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(7)
plt.style.use('default')

## 1. Create a tiny 2D dataset
A VAE learns a latent distribution, not just a compressed code.

In [ ]:
n = 120
cluster_a = np.random.normal(loc=[-2, 0], scale=0.35, size=(n // 2, 2))
cluster_b = np.random.normal(loc=[2, 0], scale=0.35, size=(n // 2, 2))
X = np.vstack([cluster_a, cluster_b])
df = pd.DataFrame(X, columns=['x1', 'x2'])
display(df.head())
fig, ax = plt.subplots(figsize=(5, 3))
ax.scatter(X[:, 0], X[:, 1], alpha=0.7)
ax.set_title('Toy data with two clusters')
ax.set_xlabel('x1')
ax.set_ylabel('x2')
plt.show()

## 2. Encode each point into mean and log-variance
This is a tiny hand-built encoder so we can inspect the quantities.

In [ ]:
W_mu = np.array([[0.6], [0.1]])
b_mu = np.array([[0.0]])
W_logvar = np.array([[-0.2], [0.1]])
b_logvar = np.array([[-1.0]])
mu = X @ W_mu + b_mu
logvar = X @ W_logvar + b_logvar
sigma = np.exp(0.5 * logvar)
encoded = pd.DataFrame({'x1': X[:, 0], 'x2': X[:, 1], 'mu': mu.ravel(), 'sigma': sigma.ravel()})
display(encoded.head())
print('mu shape:', mu.shape)
print('sigma shape:', sigma.shape)

## 3. Reparameterization trick
Sample epsilon from a standard normal, then set z = mu + sigma * epsilon.

In [ ]:
epsilon = np.random.normal(size=mu.shape)
z = mu + sigma * epsilon
print('First five epsilons:', epsilon[:5].ravel())
print('First five z samples:', z[:5].ravel())
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(z.ravel(), bins=25, color='#9ecae1', edgecolor='black')
ax.set_title('Sampled latent values z')
ax.set_xlabel('z')
plt.show()

## 4. Decode and compute losses
The reconstruction term asks whether decoded points match data. The KL term keeps the latent distribution near N(0,1).

In [ ]:
W_dec = np.array([[2.5, 0.0]])
b_dec = np.array([[0.0, 0.0]])
X_hat = z @ W_dec + b_dec
reconstruction_loss = np.mean((X - X_hat) ** 2)
kl_per_point = -0.5 * (1 + logvar - mu ** 2 - np.exp(logvar))
kl_loss = np.mean(kl_per_point)
elbo_loss = reconstruction_loss + kl_loss
print('reconstruction loss:', reconstruction_loss)
print('KL loss:', kl_loss)
print('negative ELBO-style loss:', elbo_loss)

fig, ax = plt.subplots(figsize=(5, 3))
ax.scatter(X[:, 0], X[:, 1], label='original', alpha=0.6)
ax.scatter(X_hat[:, 0], X_hat[:, 1], label='decoded sample', alpha=0.6)
ax.set_title('Original vs decoded points')
ax.legend()
plt.show()

## 5. Generate new samples from the prior

In [ ]:
z_prior = np.linspace(-2.5, 2.5, 80).reshape(-1, 1)
generated = z_prior @ W_dec + b_dec
fig, ax = plt.subplots(figsize=(6, 3))
ax.scatter(X[:, 0], X[:, 1], alpha=0.25, label='training data')
ax.plot(generated[:, 0], generated[:, 1], color='red', label='decoded latent sweep')
ax.set_title('Generation by sampling latent z and decoding')
ax.legend()
plt.show()

Try increasing `b_logvar` from -1.0 to 0.5. Larger variance makes samples more diverse but reconstruction noisier.